# Neurocontroller – Run a Trial

This notebook mirrors the functionality available via the EBRAINS Software Distribution (ESD).
It runs a single closed-loop trial (NEST + PyBullet via NRP) and displays the results.

> **Note:** One trial takes a few minutes to simulate depending on hardware.

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Image as IPImage

# Matplotlib inline for any ad-hoc plots
%matplotlib inline

from neurocontroller.nrp_start_sim import run_trial
from neurocontroller.config.ResultMeta import ResultMeta
from neurocontroller.config.paths import RunPaths
from neurocontroller.plant.plant_plotting import plot_plant_outputs
from neurocontroller.utils_common.draw_schema_svg import draw_schema
from neurocontroller.utils_common.results import gather_metas

## Optional: inspect default configuration

You can override parameters by editing the Pydantic models in `neurocontroller.config` before calling `run_trial()`.
For this notebook we use the defaults (GLE planner, e-prop M1, cerebellum enabled).

In [ ]:
from neurocontroller.config.MasterParams import MasterParams
from neurocontroller.config.core_models import SimulationParams

# Show default simulation timing
sim = SimulationParams()
print(f"Trial duration: {sim.duration_ms:.0f} ms ({sim.duration_s:.1f} s)")
print(f"Steps: {sim.sim_steps}")
print(f"Resolution: {sim.resolution} ms")

## Run one trial

`run_trial()` handles the full NRP loop: NEST initialization, PyBullet physics, and data saving.
The returned `run_id` can be used to chain further trials (see Notebook 02).

In [ ]:
# Run a single trial with an optional label
run_id = run_trial(parent_id="", label="notebook_demo")
print(f"\n=== Trial completed ===")
print(f"Run ID: {run_id}")

## Load result metadata

Each trial writes a small JSON manifest (`ResultMeta`) that points to the neural data, robotic data, and parameters.

In [ ]:
meta = ResultMeta.from_id(run_id)
print(f"Meta ID     : {meta.id}")
print(f"Parent      : {meta.parent or 'None (first trial)'}")
print(f"Neural data : {meta.neural}")
print(f"Robotic data: {meta.robotic}")
print(f"Params      : {meta.params}")

## Generate plots

The built-in plotting routines save figures to the run directory.
We call them here and then display the generated PNGs inline.

In [ ]:
# Plant-side plots (joint trajectories, motor commands, RMSE, etc.)
plot_plant_outputs([meta])

# Schema visualization (SVG + PNG)
draw_schema([meta])

### Display generated figures

In [ ]:
run_paths = RunPaths.from_run_id(run_id, create_if_not_present=False)

# Plant figures
plant_figs = sorted(run_paths.figures_receiver.glob("*.png"))
print(f"Found {len(plant_figs)} plant figure(s)")
for fig_path in plant_figs:
    display(IPImage(filename=str(fig_path)))

In [ ]:
# Schema overview (if template is present)
schema_png = run_paths.figures_receiver / "whole_controller_schema.png"
if schema_png.exists():
    display(IPImage(filename=str(schema_png)))
else:
    print("Schema PNG not found (template may be missing).")

## Next steps

- Use the `run_id` above as `parent_id` for the next trial to enable cerebellar learning across trials.
- Open **Notebook 02** to explore existing runs without re-simulating.